In [3]:
# ============================================================
# FINAL GIF CAPTION SYSTEM - TEMPLATE + MULTI-MODAL
# ============================================================
# This is your WORKING system from Phase 2
# It's better than the poorly-transferred LSTM
# ============================================================

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import random
from ultralytics import YOLO

print("=" * 60)
print(" FINAL GIF CAPTION SYSTEM")
print("   (Template-based with Multi-modal Features)")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# LOAD MODELS
# ============================================================

# 1. Emotion Model
emotion_groups = ['contempt', 'negative_intense', 'negative_subdued',
                 'positive_calm', 'positive_energetic', 'surprise']

class GroupedEmotionClassifier(nn.Module):
    def __init__(self, num_classes=6):
        super(GroupedEmotionClassifier, self).__init__()
        resnet = models.resnet50(weights=None)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

emotion_model = GroupedEmotionClassifier(num_classes=6)
emotion_model.load_state_dict(torch.load('best_model_grouped.pth', map_location=device))
emotion_model = emotion_model.to(device)
emotion_model.eval()

# 2. Object Detector
object_detector = YOLO('yolov8n.pt')

print(" All models loaded!")

# ============================================================
# EMOTION VOCABULARY
# ============================================================

emotion_vocabulary = {
    'positive_energetic': {
        'adjectives': ['joyful', 'happy', 'excited', 'cheerful', 'enthusiastic', 'energetic', 'elated'],
        'adverbs': ['happily', 'joyfully', 'excitedly', 'cheerfully', 'enthusiastically'],
        'verbs': ['dancing', 'jumping', 'celebrating', 'cheering', 'laughing', 'playing']
    },
    'positive_calm': {
        'adjectives': ['peaceful', 'content', 'serene', 'relaxed', 'satisfied', 'tranquil'],
        'adverbs': ['peacefully', 'calmly', 'serenely', 'contentedly', 'quietly'],
        'verbs': ['sitting', 'resting', 'smiling', 'relaxing', 'enjoying']
    },
    'negative_intense': {
        'adjectives': ['angry', 'furious', 'fearful', 'terrified', 'disgusted', 'enraged'],
        'adverbs': ['angrily', 'furiously', 'fearfully', 'terrifiedly', 'disgustedly'],
        'verbs': ['yelling', 'screaming', 'running', 'fighting', 'crying']
    },
    'negative_subdued': {
        'adjectives': ['sad', 'sorrowful', 'dejected', 'gloomy', 'melancholic', 'somber'],
        'adverbs': ['sadly', 'sorrowfully', 'dejectedly', 'gloomily', 'quietly'],
        'verbs': ['crying', 'sitting', 'looking', 'walking', 'waiting']
    },
    'surprise': {
        'adjectives': ['surprised', 'shocked', 'astonished', 'amazed', 'stunned', 'bewildered'],
        'adverbs': ['surprisingly', 'shockingly', 'astonishedly', 'amazedly'],
        'verbs': ['reacting', 'jumping', 'gasping', 'staring', 'looking']
    },
    'contempt': {
        'adjectives': ['contemptuous', 'disdainful', 'scornful', 'dismissive', 'mocking'],
        'adverbs': ['contemptuously', 'disdainfully', 'scornfully', 'dismissively'],
        'verbs': ['dismissing', 'ignoring', 'mocking', 'scoffing', 'rejecting']
    }
}

# ============================================================
# TEMPLATE CAPTION GENERATOR
# ============================================================

templates = [
    "a {adj} person {verb}",
    "someone {verb} {adv}",
    "a {adj} person {verb} with {obj}",
    "a person {verb} {adv}",
    "an animated scene of a {adj} person {verb}",
    "a {adj} moment showing someone {verb}",
    "someone feeling {adj} and {verb}",
    "a person {verb} in a {adj} way",
]

def generate_caption(emotion, objects=None):
    """Generate template-based caption with emotion and optional objects"""
    vocab = emotion_vocabulary[emotion]
    
    adj = random.choice(vocab['adjectives'])
    adv = random.choice(vocab['adverbs'])
    verb = random.choice(vocab['verbs'])
    
    # Filter out 'person' from objects
    non_person_objs = [o for o in objects if o.lower() != 'person'] if objects else []
    
    # Choose template type based on available objects
    if non_person_objs:
        # We have real objects - include them
        obj = random.choice(non_person_objs)
        templates_with_obj = [
            "a {adj} person {verb} with a {obj}",
            "someone {verb} {adv} near a {obj}",
            "a {adj} person {verb} beside a {obj}",
            "a person {verb} {adv} with a {obj}",
        ]
        template = random.choice(templates_with_obj)
        caption = template.format(adj=adj, adv=adv, verb=verb, obj=obj)
    else:
        # No objects - use simple templates
        templates_simple = [
            "a {adj} person {verb}",
            "someone {verb} {adv}",
            "a person {verb} in a {adj} way",
            "an animated scene of a {adj} person {verb}",
            "a {adj} moment showing someone {verb}",
            "someone feeling {adj} while {verb}",
            "a person {verb} {adv}",
        ]
        template = random.choice(templates_simple)
        caption = template.format(adj=adj, adv=adv, verb=verb)
    
    return caption

# ============================================================
# COMPLETE PIPELINE
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_middle_frame(gif_path):
    try:
        with Image.open(gif_path) as gif:
            n_frames = 0
            try:
                while True:
                    gif.seek(n_frames)
                    n_frames += 1
            except EOFError:
                pass
            gif.seek(n_frames // 2)
            return gif.convert('RGB')
    except:
        return None

def detect_emotion(gif_path):
    frame = extract_middle_frame(gif_path)
    if frame is None:
        return None, None
    
    frame_tensor = transform(frame).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = emotion_model(frame_tensor)
        probs = torch.softmax(output, dim=1)
        idx = torch.argmax(probs, dim=1).item()
        confidence = probs[0, idx].item()
    
    return emotion_groups[idx], confidence

def detect_objects(gif_path):
    frame = extract_middle_frame(gif_path)
    if frame is None:
        return []
    
    results = object_detector(frame, verbose=False)
    objects = []
    
    for result in results:
        boxes = result.boxes
        for box in boxes:
            conf = float(box.conf[0])
            if conf >= 0.3:
                class_id = int(box.cls[0])
                class_name = result.names[class_id]
                objects.append(class_name)
    
    return list(set(objects))[:3]  # Top 3 unique objects

def generate_gif_caption(gif_path):
    """Complete pipeline"""
    # Detect emotion
    emotion, confidence = detect_emotion(gif_path)
    if emotion is None:
        return "Unable to process GIF", None, None, []
    
    # Detect objects
    objects = detect_objects(gif_path)
    
    # Generate caption
    caption = generate_caption(emotion, objects)
    
    return caption, emotion, confidence, objects

print("\n Pipeline ready!")

 FINAL GIF CAPTION SYSTEM
   (Template-based with Multi-modal Features)
 All models loaded!

 Pipeline ready!


In [4]:
# ============================================================
# TEST ON GIFGIF DATASET
# ============================================================

test_df = pd.read_csv('test_grouped.csv')

print("\n" + "=" * 60)
print(" TESTING FINAL SYSTEM")
print("=" * 60)

samples = test_df.sample(10, random_state=42)

for idx, (_, row) in enumerate(samples.iterrows()):
    gif_path = row['gif_path']
    true_emotion = row['emotion_group']
    gif_id = row['gif_id']
    
    caption, pred_emotion, confidence, objects = generate_gif_caption(gif_path)
    
    match = "" if pred_emotion == true_emotion else ""
    
    print(f"\n{'─' * 60}")
    print(f"  GIF #{idx+1}: {gif_id}")
    print(f"  {match} True Emotion:      {true_emotion}")
    print(f"     Predicted Emotion:  {pred_emotion} ({confidence:.1%})")
    print(f"   Objects:           {', '.join(objects) if objects else 'None'}")
    print(f"   Caption:           {caption}")
    print(f"{'─' * 60}")

print("\n" + "=" * 60)
print(" TESTING COMPLETE!")
print("=" * 60)


 TESTING FINAL SYSTEM

────────────────────────────────────────────────────────────
  GIF #1: HSNAwKc63r9lK
   True Emotion:      negative_intense
     Predicted Emotion:  negative_intense (56.1%)
   Objects:           person
   Caption:           a person fighting in a angry way
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  GIF #2: 9YdllfJmcs5RS
   True Emotion:      negative_intense
     Predicted Emotion:  negative_intense (96.6%)
   Objects:           person
   Caption:           someone feeling disgusted while screaming
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  GIF #3: aF8nZCmUEgmBO
   True Emotion:      surprise
     Predicted Emotion:  negative_intense (45.4%)
   Objects:           person
   Caption:           a angry person running
────────────────────────────────────────────────────────────

─────────────────────────

In [5]:
# ============================================================
# FULL TEST SET EVALUATION
# ============================================================

import pandas as pd
from tqdm import tqdm

test_df = pd.read_csv('test_grouped.csv')

print("=" * 60)
print(" FULL TEST SET EVALUATION")
print("=" * 60)
print(f"Total test GIFs: {len(test_df)}\n")
print("Processing...\n")

results = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Generating captions"):
    gif_path = row['gif_path']
    true_emotion = row['emotion_group']
    gif_id = row['gif_id']
    
    caption, pred_emotion, confidence, objects = generate_gif_caption(gif_path)
    
    # Check if caption contains emotion word
    vocab = emotion_vocabulary.get(pred_emotion, {})
    all_emotion_words = (
        vocab.get('adjectives', []) + 
        vocab.get('adverbs', []) + 
        vocab.get('verbs', [])
    )
    has_emotion_word = any(word in caption.lower() for word in all_emotion_words)
    
    results.append({
        'gif_id': gif_id,
        'true_emotion': true_emotion,
        'pred_emotion': pred_emotion,
        'confidence': confidence,
        'objects': ', '.join(objects) if objects else '',
        'num_objects': len([o for o in objects if o.lower() != 'person']) if objects else 0,
        'caption': caption,
        'emotion_correct': pred_emotion == true_emotion,
        'has_emotion_word': has_emotion_word
    })

results_df = pd.DataFrame(results)

# Calculate metrics
emotion_accuracy = results_df['emotion_correct'].mean()
emotion_word_inclusion = results_df['has_emotion_word'].mean()
avg_confidence = results_df['confidence'].mean()
captions_with_objects = (results_df['num_objects'] > 0).mean()

print("\n" + "=" * 60)
print(" RESULTS SUMMARY")
print("=" * 60)
print(f"Total GIFs processed:        {len(results_df)}")
print(f"Emotion accuracy:            {emotion_accuracy:.2%}")
print(f"Average confidence:          {avg_confidence:.2%}")
print(f"Emotion word inclusion:      {emotion_word_inclusion:.2%} ⭐")
print(f"Captions with objects:       {captions_with_objects:.2%}")
print()

# Emotion-wise breakdown
print(" Emotion-wise Accuracy:")
print("-" * 60)
for emotion in emotion_groups:
    subset = results_df[results_df['true_emotion'] == emotion]
    acc = subset['emotion_correct'].mean() if len(subset) > 0 else 0
    count = len(subset)
    print(f"  {emotion:20s}: {acc:.2%} ({count} samples)")

print("\n" + "=" * 60)

# Save results
results_df.to_csv('full_test_results.csv', index=False)
print(f" Results saved to: full_test_results.csv")

# Get best examples (high confidence + correct emotion)
best_examples = results_df[
    (results_df['emotion_correct'] == True) & 
    (results_df['confidence'] > 0.85)
].sort_values('confidence', ascending=False).head(15)

print("\n" + "=" * 60)
print(" TOP 15 EXAMPLES FOR THESIS")
print("=" * 60)
for idx, row in best_examples.iterrows():
    print(f"\nGIF: {row['gif_id']}")
    print(f"  Emotion: {row['pred_emotion']} ({row['confidence']:.1%} confidence)")
    print(f"  Objects: {row['objects'] if row['objects'] else 'None'}")
    print(f"  Caption: {row['caption']}")

print("\n" + "=" * 60)
print(" EVALUATION COMPLETE!")
print("=" * 60)

 FULL TEST SET EVALUATION
Total test GIFs: 889

Processing...



Generating captions: 100%|██████████| 889/889 [05:53<00:00,  2.52it/s]


 RESULTS SUMMARY
Total GIFs processed:        889
Emotion accuracy:            35.55%
Average confidence:          70.64%
Emotion word inclusion:      100.00% ⭐
Captions with objects:       31.95%

 Emotion-wise Accuracy:
------------------------------------------------------------
  contempt            : 7.69% (26 samples)
  negative_intense    : 38.81% (219 samples)
  negative_subdued    : 42.86% (147 samples)
  positive_calm       : 19.18% (146 samples)
  positive_energetic  : 43.91% (271 samples)
  surprise            : 23.75% (80 samples)

 Results saved to: full_test_results.csv

 TOP 15 EXAMPLES FOR THESIS

GIF: 13k4VSc3ngLPUY
  Emotion: positive_energetic (99.9% confidence)
  Objects: None
  Caption: a excited moment showing someone cheering

GIF: hKtgxn739bVNC
  Emotion: negative_subdued (99.9% confidence)
  Objects: person
  Caption: a melancholic person waiting

GIF: 138Gw4DrgWn9Mk
  Emotion: positive_energetic (99.7% confidence)
  Objects: dog
  Caption: a enthusiastic per